# Training maia on Google Cloud (L4)

This notebook runs **on your machine**. It does not train here: it prepares the dataset, uploads it to Cloud
Storage and starts a VM with an **NVIDIA L4** GPU that trains with **QLoRA (Unsloth)** and shuts itself down when
done.

Flow:

1. **Configuration** — model, dataset variant (K, KP, KPS) and hyperparameters.
2. **Setup** (first time only) — APIs, bucket and service account.
3. **Export** the dataset with `scripts/exporta_dataset.py`.
4. **Upload** the run and **start** the VM.
5. **Follow** the log until it finishes.
6. **Download** the adapter, metrics and samples to `runs/<run>/`.
7. **Publish** the adapter to Hugging Face.
8. **Delete** the VM.

Local requirements: `gcloud` authenticated with an account that has permissions on the `maia-andorra` project, and
`HF_TOKEN` (a Hugging Face token with write access) in the environment for step 7.

Cost: the VM is **spot** by default (much cheaper, but Google can stop it). If it stops, the «Resume» cell starts it
again and training continues from the last checkpoint. There is a hard limit of hours (`MAX_HOURS`): after that the
VM stops anyway.

## 1. Configuration

In [ ]:
import datetime, json, os, shutil, subprocess, sys, time
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "training" else Path.cwd()
assert (ROOT / "scripts" / "exporta_dataset.py").exists(), f"Open the notebook from the maia repo root (now: {ROOT})"

PROJECT = "maia-andorra"
REGION = "europe-west4"                      # bucket region
# Zones with L4s where we have quota (1 GPU per region; the global limit is also 1). Tried in order:
# L4s often run out in a whole region (it happened on 2026-10-27 across europe-west4).
ZONES = [
    "europe-west4-a", "europe-west4-b", "europe-west4-c",
    "europe-west1-b", "europe-west1-c", "europe-west3-a", "europe-west3-b",
    "europe-west2-a", "europe-west2-b", "europe-west6-b", "europe-west6-c",
    "europe-west9-a", "europe-west9-b", "europe-west9-c",
    "europe-southwest1-a", "europe-southwest1-b", "europe-southwest1-c",
]
BUCKET = "maia-andorra-entrenaments"
SERVICE_ACCOUNT = f"maia-entrenament@{PROJECT}.iam.gserviceaccount.com"

MODELS = {
    "e4b": "unsloth/gemma-4-E4B-it-unsloth-bnb-4bit",
    "31b": "unsloth/gemma-4-31B-it-unsloth-bnb-4bit",   # QLoRA ~22 GB according to Unsloth: tight on the L4
}
MODEL = "e4b"
VARIANT = "K"            # K = knowledge · KP = + real Parla · KPS = + synthetic Parla
# True = 30 steps to check that everything works; False = full training.
# Without opening the notebook: MAIA_TEST=0 jupyter nbconvert --to notebook --execute training/train.ipynb
TEST = os.environ.get("MAIA_TEST", "1") != "0"
SPOT = True
MAX_HOURS = 10

CONFIG = {
    "model": MODELS[MODEL],
    "variant": VARIANT,
    "chat_template": "gemma-4" if MODEL == "e4b" else "gemma-4-thinking",
    "max_seq_length": 2048,
    "lora_r": 32,
    "lora_alpha": 32,
    "epochs": 2,
    "learning_rate": 2e-4,
    "batch_size": 4,
    "grad_accum": 4,          # effective batch = 16
    "eval_steps": 200,
    "seed": 3407,
    "export_gguf": False,
    # Unsloth with Gemma 4: per-block compilation breaks gradient checkpointing
    # (CheckpointError on the first step, unsloth#14063).
    "env": {"UNSLOTH_BLOCK_COMPILE": "0"},
    # Sample prompts stay in Catalan: they are what the model is trained to answer.
    "prompts": [
        "Què és el Consell de la Terra?",
        "Quines són les set parròquies d'Andorra?",
        "Explica'm què era la Querimònia.",
        "Què es menja per Sant Jordi a Andorra?",
        "Com va anar la reforma constitucional del 1993?",
    ],
}
if TEST:
    CONFIG["max_steps"] = 30
    CONFIG["eval_steps"] = 15

RUN = f"{datetime.datetime.now():%Y%m%d-%H%M}-{MODEL}-{VARIANT.lower()}" + ("-test" if TEST else "")
CONFIG["run"] = RUN
VM = f"maia-{RUN}"
GS = f"gs://{BUCKET}/runs/{RUN}"
print("Run:", RUN)
print("VM: ", VM)

In [ ]:
def sh(cmd, check=True, quiet=False):
    """Runs a command and returns its output. Always against PROJECT, never gcloud's default project."""
    if not quiet:
        print("$", " ".join(cmd))
    res = subprocess.run(cmd, capture_output=True, text=True)
    if check and res.returncode != 0:
        raise RuntimeError(res.stderr.strip() or res.stdout.strip())
    return res.stdout.strip()

def gcloud(*args, **kw):
    return sh(["gcloud", *args, f"--project={PROJECT}"], **kw)

## 2. Setup (first time only)

Enables the APIs, creates the bucket and a dedicated service account for the VM, with access to this bucket only.
Safe to re-run: whatever already exists is left untouched.

In [ ]:
gcloud("services", "enable", "compute.googleapis.com", "storage.googleapis.com", "iam.googleapis.com")

if gcloud("storage", "buckets", "list", f"--filter=name={BUCKET}", "--format=value(name)", quiet=True) != BUCKET:
    gcloud("storage", "buckets", "create", f"gs://{BUCKET}", f"--location={REGION}", "--uniform-bucket-level-access")

sa_name = SERVICE_ACCOUNT.split("@")[0]
if not gcloud("iam", "service-accounts", "list", f"--filter=email={SERVICE_ACCOUNT}", "--format=value(email)", quiet=True):
    gcloud("iam", "service-accounts", "create", sa_name, "--display-name=maia training")
    time.sleep(10)  # the account takes a few seconds to exist for IAM

gcloud("storage", "buckets", "add-iam-policy-binding", f"gs://{BUCKET}",
       f"--member=serviceAccount:{SERVICE_ACCOUNT}", "--role=roles/storage.objectAdmin", quiet=True)
print("Setup done.")

## 3. Export the dataset

Regenerates `export/K`, `export/KP` and `export/KPS` from `training-data/datasets/`. Every conversation carries the
fixed system message from `training-data/missatge-sistema.txt`, and the cards reserved for the benchmark are left out.

In [ ]:
print(sh([sys.executable, str(ROOT / "scripts" / "exporta_dataset.py")]) or "Export done.")
print((ROOT / "export" / "INFORME.md").read_text(encoding="utf-8")[:1500])

## 4. Upload the run and start the VM

In [ ]:
stage = ROOT / "runs" / RUN / "input"
if stage.exists():
    shutil.rmtree(stage)
(stage / "data").mkdir(parents=True)
for name in ("train.jsonl", "valid.jsonl"):
    shutil.copy(ROOT / "export" / VARIANT / name, stage / "data" / name)
shutil.copy(ROOT / "training" / "train.py", stage / "train.py")
(stage / "config.json").write_text(json.dumps(CONFIG, ensure_ascii=False, indent=2), encoding="utf-8")
gcloud("storage", "rsync", "-r", str(stage), f"{GS}/input")
print("Uploaded to", GS)

In [ ]:
def create_vm(zone):
    args = [
        "compute", "instances", "create", VM, f"--zone={zone}",
        "--machine-type=g2-standard-8",          # 1x L4 (24 GB), 8 vCPU, 32 GB RAM
        "--maintenance-policy=TERMINATE",
        "--image-family=common-cu129-ubuntu-2204-nvidia-580",
        "--image-project=deeplearning-platform-release",
        "--boot-disk-size=200GB", "--boot-disk-type=pd-balanced",
        f"--service-account={SERVICE_ACCOUNT}", "--scopes=cloud-platform",
        f"--metadata=bucket={BUCKET},run={RUN},install-nvidia-driver=True",
        f"--metadata-from-file=startup-script={ROOT / 'training' / 'vm-startup.sh'}",
        f"--labels=maia=training,run={RUN}",
        f"--max-run-duration={MAX_HOURS}h", "--instance-termination-action=STOP",
    ]
    if SPOT:
        args.append("--provisioning-model=SPOT")
    return gcloud(*args)

def create_anywhere():
    """Tries each zone until one has a free L4. Returns the zone or None."""
    for zone in ZONES:
        try:
            create_vm(zone)
            return zone
        except RuntimeError as exc:
            reason = "no capacity" if "EXHAUSTED" in str(exc) or "STOCKOUT" in str(exc) else str(exc).splitlines()[-1]
            print(f"{zone}: {reason}")
    return None

ZONE = create_anywhere()
assert ZONE, "No zone has capacity right now. Try again later or set SPOT = False."
print(f"VM {VM} started in {ZONE}")

## 5. Follow the training

Shows the VM status and the last log lines every minute, until the run finishes (`DONE` or `FAILED`) or the VM
stops. The first time, the VM takes ~10 minutes to install Unsloth before it starts training.

Note: with Gemma 4 E4B, Unsloth warns that an initial loss of 13–15 is normal. What matters is that it goes down.

In [ ]:
def status():
    vm = gcloud("compute", "instances", "describe", VM, f"--zone={ZONE}", "--format=value(status)", check=False, quiet=True)
    marks = gcloud("storage", "ls", f"{GS}/", check=False, quiet=True)
    done = f"{GS}/DONE" in marks
    failed = f"{GS}/FAILED" in marks
    return vm or "DELETED", done, failed

def tail(n=15):
    log = gcloud("storage", "cat", f"{GS}/train.log", check=False, quiet=True)
    return "\n".join(log.splitlines()[-n:])

while True:
    vm, done, failed = status()
    print(f"--- {datetime.datetime.now():%H:%M} · VM {vm} · {'DONE' if done else 'FAILED' if failed else 'running'}")
    print(tail())
    if done or failed or vm in ("TERMINATED", "STOPPED", "DELETED"):
        break
    time.sleep(60)

### Resume (only if the spot VM stopped before finishing)

If the status is `TERMINATED` without `DONE` or `FAILED`, Google has reclaimed the spot VM. This cell starts it
again; if the zone has no capacity, it deletes it and creates a new one in the first free zone of `ZONES`. All state
lives in the bucket, so the startup script resumes from the last uploaded checkpoint (every 5 minutes).
Then go back to the previous cell.

In [ ]:
vm, done, failed = status()
if vm == "TERMINATED" and not (done or failed):
    try:
        gcloud("compute", "instances", "start", VM, f"--zone={ZONE}")
        print("VM restarted.")
    except RuntimeError as exc:
        print(f"Cannot restart in {ZONE} ({str(exc).splitlines()[-1]}). Recreating it in another zone.")
        gcloud("compute", "instances", "delete", VM, f"--zone={ZONE}", "--quiet")
        ZONE = create_anywhere()
        assert ZONE, "No zone has capacity right now. Try again later."
        print(f"VM {VM} recreated in {ZONE}")
else:
    print(f"Not needed: VM {vm}, DONE={done}, FAILED={failed}")

## 6. Download the results

In [ ]:
dest = ROOT / "runs" / RUN / "output"
dest.mkdir(parents=True, exist_ok=True)
gcloud("storage", "cp", f"{GS}/train.log", str(dest / "train.log"), check=False)
vm, done, failed = status()
if not done:
    # Without DONE there is no adapter: only the log is downloaded to see what happened.
    print(f"The run did not finish cleanly (VM {vm}, FAILED={failed}). Last log lines:")
    print(tail(40))
    raise SystemExit("Check runs/<run>/output/train.log before launching it again.")
# Intermediate checkpoints are not needed: the final adapter is in output/adapter.
gcloud("storage", "rsync", "-r", "--exclude=^checkpoints/", f"{GS}/output", str(dest))
metrics = json.loads((dest / "metrics.json").read_text(encoding="utf-8"))
print("Final validation loss:", metrics["final_eval"].get("eval_loss"))
evals = [(h["step"], round(h["eval_loss"], 4)) for h in metrics["log_history"] if "eval_loss" in h]
print("Validation curve (step, loss):", evals)

In [ ]:
from IPython.display import Markdown, display
samples = dest / "samples.md"
display(Markdown(samples.read_text(encoding="utf-8")) if samples.exists() else "No samples.")

## 7. Publish the adapter to Hugging Face

Uploads the adapter, metrics, samples and a generated model card to a **private** model repo `<user>/maia-<run>`
(`training/upload_hf.py`). Needs `HF_TOKEN` with write access in the environment.

In [ ]:
if os.environ.get("HF_TOKEN"):
    print(sh([sys.executable, str(ROOT / "training" / "upload_hf.py"), str(ROOT / "runs" / RUN)]))
else:
    print("HF_TOKEN is not set: skipped. Run training/upload_hf.py runs/<run> by hand.")

## 8. Delete the VM

A stopped VM no longer pays for the GPU, but the 200 GB disk still costs. Delete it once you have the results.
The run files stay in the bucket.

In [ ]:
gcloud("compute", "instances", "delete", VM, f"--zone={ZONE}", "--quiet")
print("VM deleted.")

## Appendix — existing VMs and runs

In [ ]:
print(gcloud("compute", "instances", "list", "--filter=labels.maia:*",
             "--format=table(name,zone.basename(),status,scheduling.provisioningModel)", quiet=True) or "No VMs.")
print(gcloud("storage", "ls", f"gs://{BUCKET}/runs/", check=False, quiet=True) or "No runs.")